In [5]:
# ========================================
# TinyML - Predict Wind Energy Production
# Dataset: wind_power_generation.csv
# Target: Predict Power using time and weather features
# ========================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score

# Load the dataset from local or working directory
data = pd.read_csv('wind_power_generation.csv')

# Show the first few rows to understand the structure
data.head()



,Time,temperature_2m,relativehumidity_2m,dewpoint_2m,windspeed_10m,windspeed_100m,winddirection_10m,winddirection_100m,windgusts_10m,Power
0,2017-01-02 00:00:00,28.5,85,24.5,1.44,1.26,146,162,1.4,0.1635
1,2017-01-02 01:00:00,28.4,86,24.7,2.06,3.99,151,158,4.4,0.1424
2,2017-01-02 02:00:00,26.8,91,24.5,1.30,2.78,148,150,3.2,0.1214
3,2017-01-02 03:00:00,27.4,88,24.3,1.30,2.69,58,105,1.6,0.1003
4,2017-01-02 04:00:00,27.3,88,24.1,2.47,4.43,58,84,4.0,0.0793


In [6]:
# ========================================
# Train ML Model
# ========================================

# Parse the time column and extract features
data['Time'] = pd.to_datetime(data['Time'])

# Resample to 15-minute intervals and interpolate linearly
data = data.set_index('Time').resample('15min').interpolate(method='linear').reset_index()

# Remove some data to reduce dimensions
data = data[data['Time'] <= '2018-01-01'].copy()

data['Month'] = data['Time'].dt.month
data['Day'] = data['Time'].dt.day

# Continuous hour (e.g., 01:15 → 1.25)
data['HourDecimal'] = (
    data['Time'].dt.hour
  + data['Time'].dt.minute / 60.0
  + data['Time'].dt.second / 3600.0
)

# Keep only daytime data (2AM to 7AM)
data = data[(data['Time'].dt.hour <= 2) | (data['Time'].dt.hour >= 7)]

# Predict 20 minutes into the future (i.e., one step ahead)
data['TargetPower'] = data['Power'].shift(-1)

# Drop rows with missing values if any
data = data.dropna()

features = ['Month', 'Day', 'HourDecimal']

X = data[features]
y = data['TargetPower']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaler mean:", scaler.mean_)
print("Scaler scale:", scaler.scale_)


Scaler mean: [ 6.52755838 15.81850962 13.26916638]
Scaler scale: [3.44633796 8.76325198 6.7547069 ]


In [7]:
# ========================================
# Train Random Forest Model
# ========================================
model = RandomForestRegressor(n_estimators=5, max_depth=12, random_state=42)
model.fit(X_train_scaled, y_train)

# Make predictions
y_pred = model.predict(X_test_scaled)

# Evaluation
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse:.4f}")
print(f"R² Score: {r2:.4f}")

Mean Squared Error: 0.0060
R² Score: 0.9226


In [8]:
# ========================================
# Binned Accuracy
# ========================================

bins = np.histogram_bin_edges(y, bins=5)

y_test_binned = np.digitize(y_test, bins)
y_pred_binned = np.digitize(y_pred, bins)

acc = accuracy_score(y_test_binned, y_pred_binned)

print(f"Binned Accuracy: {acc:.2f}")

Binned Accuracy: 0.79


In [9]:
# ========================================
# Export Model to TinyML (C header)
# ========================================

!pip install emlearn

import emlearn

# ========================================
# Export Model to C (.h) for TinyML
# ========================================
cmodel = emlearn.convert(model, method='inline', dtype='float')
cmodel.save(file='./wind_power_prediction.h', name='wind_power_prediction')

print("Model exported to wind_power_prediction.h")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.4/88.4 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Using cached pybind11-2.13.6-py3-none-any.whl.metadata (9.5 kB)
Using cached pybind11-2.13.6-py3-none-any.whl (243 kB)
  Created wheel for emlearn: filename=emlearn-0.21.2-cp311-cp311-linux_x86_64.whl size=1765772 sha256=b42b1286f179a085021e89cbfafcf87ed034229540684444ab2bb94ceccbb316
  Stored in directory: /root/.cache/pip/wheels/14/d7/f5/a8dfc99837cf8209704b272811522b90cbdabb88ccd4df6321
Successfully built emlearn
Model exported to wind_power_prediction.h
